# CrewAI — roles, tasks, crews

CrewAI encodes an agent's persona as a `role / goal / backstory` triple. Work is a `Task` with a description and an expected output. A `Crew` orchestrates agents over tasks. The framework's value is *legibility* — non-engineers can read a crew definition and follow what's happening.

## Canonical code (with `crewai` installed)

```python
from crewai import Agent, Task, Crew, Process
from crewai.tools import tool
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from task import search_corpus, fetch_paper, cite

@tool('search_corpus')
def search_corpus_tool(query: str) -> list[dict]:
    '''Search the arxiv corpus and return the top 2 hits.'''
    return search_corpus(query, k=2)

@tool('cite_claim')
def cite_tool(arxiv_id: str, claim: str) -> dict:
    '''Verify a claim against a paper's abstract.'''
    return cite(arxiv_id, claim)

researcher = Agent(
    role='Research Analyst',
    goal='Find the most relevant paper for the user question.',
    backstory='You read arxiv compulsively. Precision over breadth.',
    tools=[search_corpus_tool],
)
summariser = Agent(
    role='Citation-First Summariser',
    goal='Produce a 2-sentence answer that cites every claim.',
    backstory='You despise unsourced statements.',
    tools=[cite_tool],
)

task1 = Task(description='Find the paper that answers: {question}', agent=researcher, expected_output='An arxiv_id and snippet.')
task2 = Task(description='Write the answer with citations.', agent=summariser, expected_output='2-sentence answer with [arxiv_id] tags.')

crew = Crew(agents=[researcher, summariser], tasks=[task1, task2], process=Process.sequential)
result = crew.kickoff(inputs={'question': 'What is RA-MoE?'})
```

In [ ]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'shared').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / '03-agentic-frameworks'))
if not (os.getenv('OPENAI_API_KEY') or os.getenv('ANTHROPIC_API_KEY')):
    os.environ.setdefault('LLM_CACHE_ONLY', '1')
from task import get_question
from eval import crewai_solve
trace = crewai_solve(get_question('q01'))
for step in trace.steps:
    print(f"{step.role}: {step.name or ''}  {step.output_summary or step.content or ''}"[:140])

## Tradeoffs

* **+ Legible** — role/goal/backstory reads like an org chart.
* **+ Sequential and hierarchical processes** built in (no graph wiring).
* **− More LLM calls per request** — each agent typically does its own reasoning before tool use.
* **− Less typing/streaming maturity** than LangGraph or Pydantic AI.